# V2 — Feature Engineering — 1H BTC/USDT

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = (Path.cwd() / "../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
plt.style.use("seaborn-v0_8-darkgrid")
print(f"Project root: {PROJECT_ROOT}")


## 1. Load Feature Data

In [ ]:
train = pd.read_csv(PROJECT_ROOT / "data/features/train.csv", index_col=0, parse_dates=True)
val   = pd.read_csv(PROJECT_ROOT / "data/features/val.csv",   index_col=0, parse_dates=True)
test  = pd.read_csv(PROJECT_ROOT / "data/features/test.csv",  index_col=0, parse_dates=True)
print(f"Columns: {train.columns.tolist()}")
print(f"Train: {len(train):,} rows  {train.index[0].date()} → {train.index[-1].date()}")
print(f"Val  : {len(val):,} rows")
print(f"Test : {len(test):,} rows")


## 2. Feature Statistics

In [ ]:
train[["close","sma_10","sma_50","rsi","momentum_5"]].describe().round(4)


## 3. SMA Crossovers (Train)

In [ ]:
# Show last 2000 hours of training data for readability
sample = train.tail(2000)
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(sample.index, sample["close"],  label="Close",  linewidth=1.0, color="black")
ax.plot(sample.index, sample["sma_10"], label="SMA-10", linewidth=1.5, color="steelblue", linestyle="--")
ax.plot(sample.index, sample["sma_50"], label="SMA-50", linewidth=1.5, color="darkorange", linestyle="--")
ax.set_title("SMA-10 / SMA-50 Crossovers — Training Data (last 2,000 hours, V2)", fontsize=13)
ax.set_ylabel("Price (USDT)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.legend()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "results/figures/v2_02_sma_crossovers.png", dpi=150)
plt.show()


## 4. RSI

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
sample = train.tail(2000)
ax1.plot(sample.index, sample["close"], color="black", linewidth=1.0)
ax1.set_ylabel("Close (USDT)")
ax1.set_title("RSI-14 — Training Data (last 2,000 hours, V2)")
ax2.plot(sample.index, sample["rsi"], color="purple", linewidth=1.0)
ax2.axhline(70, color="red",   linewidth=0.8, linestyle="--", label="Overbought (70)")
ax2.axhline(30, color="green", linewidth=0.8, linestyle="--", label="Oversold (30)")
ax2.fill_between(sample.index, 70, sample["rsi"].clip(70), alpha=0.2, color="red")
ax2.fill_between(sample.index, sample["rsi"].clip(upper=30), 30, alpha=0.2, color="green")
ax2.set_ylabel("RSI")
ax2.set_ylim(0, 100)
ax2.legend(loc="upper right", fontsize=8)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "results/figures/v2_02_rsi.png", dpi=150)
plt.show()


## 5. Momentum

In [ ]:
fig, ax = plt.subplots(figsize=(16, 3))
sample = train.tail(2000)
ax.bar(sample.index, sample["momentum_5"], color=["seagreen" if v>=0 else "crimson" for v in sample["momentum_5"]], width=0.04, alpha=0.7)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("5-Hour Momentum — Training Data (last 2,000 hours, V2)")
ax.set_ylabel("Momentum (%)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "results/figures/v2_02_momentum.png", dpi=150)
plt.show()


## 6. Feature Correlation

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = train[["open","high","low","close","volume","sma_10","sma_50","rsi","momentum_5"]].corr()
sns_import = __import__("seaborn")
sns_import.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title("Feature Correlation — Training Data (V2)")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "results/figures/v2_02_correlation.png", dpi=150)
plt.show()


## 7. Normalised Data Check

In [ ]:
train_norm = pd.read_csv(PROJECT_ROOT / "data/normalized/train.csv", index_col=0, parse_dates=True)
val_norm   = pd.read_csv(PROJECT_ROOT / "data/normalized/val.csv",   index_col=0, parse_dates=True)
test_norm  = pd.read_csv(PROJECT_ROOT / "data/normalized/test.csv",  index_col=0, parse_dates=True)
print("Normalised range check (OHLCV + SMA — should be in [0, 1] for train):")
for col in ["open","high","low","close","volume","sma_10","sma_50"]:
    mn, mx = train_norm[col].min(), train_norm[col].max()
    print(f"  {col:12s}: [{mn:.4f}, {mx:.4f}]")
print()
print("Test set OHLCV may exceed 1.0 (prices above training max — clipped in env):")
for col in ["open","close"]:
    mx = test_norm[col].max()
    print(f"  test {col:6s} max: {mx:.4f}")
